# `game_state` -- 24-Dim Per-Ball Feature Vector

**Ported from `project_hrishav/feature_engineering.py`**'s `game_state_vector()` / `build_game_state_matrix()`, feeding `src/transformer_model.py`'s causal Transformer (the secondary alternative win-probability model).

**The adaptation challenge:** hrishav's original row-by-row builder assumed `data_loader.py` had *already* computed a long list of cumulative per-ball state columns (score before this ball, wickets before this ball, partnership runs, in-innings batter strike rate, etc.) while parsing raw Cricsheet JSON. `ipl_data.xlsx` doesn't carry any of those pre-computed columns -- so this module computes all of them itself, from scratch, using **vectorised pandas groupby/cumsum operations** (much faster than hrishav's original row-by-row Python loop, and necessary since we're computing from raw columns instead of reading pre-computed ones).

**A real bug was found and fixed while validating this module against real data** (see `scripts/validate_game_state.py` and `tests/test_game_state.py`) -- both are called out inline below at the exact lines they affect.

## The 24 features, in order

| # | Feature | Normalisation |
|---|---|---|
| 0 | `over` | / 20 |
| 1 | `legal_balls_total` | / 120 |
| 2 | innings==2 flag | -- |
| 3-5 | phase one-hot | 0=Powerplay, 1=Middle, 2=Death |
| 6 | `score_before` | / 250 |
| 7 | `wickets_before` | / 10 |
| 8 | `run_rate` | clipped at 15/over |
| 9 | runs in the last completed over | / 30 |
| 10 | wickets in the last 5 overs | / 5 |
| 11 | `partnership_runs` | / 150 |
| 12 | `partnership_balls` | / 60 |
| 13 | `batter_sr_innings` | / 200 |
| 14 | `batter_balls_innings` | / 120 |
| 15 | `bowler_econ_innings` | clipped at 20/over |
| 16 | `bowler_wkts_innings` | / 10 |
| 17 | boundary rate so far this innings | -- |
| 18 | dot rate so far this innings | -- |
| 19 | `runs_required` (innings 2 only) | / 200 |
| 20 | `balls_remaining` (innings 2 only) | / 120 |
| 21 | `required_rr` (innings 2 only) | clipped at 36/over |
| 22 | `toss_won_bat` | 1 if toss winner chose to bat |
| 23 | innings==2 flag (duplicate of #2) | kept for parity with the original |

In [1]:
import numpy as np
import pandas as pd

GAME_STATE_DIM = 24

## Step 1: sort deliveries and fix the "legal ball" definition

Every downstream cumulative statistic depends on processing deliveries in the correct chronological order within each innings, hence the `sort_values(["match_id", "innings", "over", "ball"])` up front.

**The bug that was here, and the fix:** hrishav's original Cricsheet-based convention treats a delivery as "legal" (counts toward the 120-ball innings limit) if it isn't a wide -- no-balls still count as legal, since "the over still advances" in their comment. But `ipl_data.xlsx` follows a **different** convention: empirically, across all 273,503 deliveries in the dataset, gagan's own `team_balls` column increments if and only if the delivery is **neither** a wide **nor** a no-ball -- zero exceptions. Using hrishav's original definition here would make this module's `legal_balls_total` silently disagree with `team_balls`/`crr`/`rrr` as used everywhere else in `src/pipeline.py`. Fixed by matching gagan's own convention.

## Step 2: cumulative "before this ball" state, and the second bug

For every quantity that needs to represent "the state right before this ball happened" (so the model can't cheat by seeing the outcome it's trying to predict), the pattern is: take a cumulative sum *through* this row, then subtract this row's own contribution.

`legal_balls_total` is kept as an **inclusive** running ball count (matching `team_balls`'s own semantics, so the two can be compared 1:1), while a new `legal_balls_before` (**exclusive**, i.e. genuinely "before this ball") is introduced specifically for ratios that need to pair with `score_before`/`wickets_before` (both already exclusive by construction).

**The second bug this module had:** `run_rate`, `boundary_rate`/`dot_rate`, and `balls_remaining` originally divided by `legal_balls_total` (inclusive) instead of `legal_balls_before` (exclusive) -- mixing an "exclusive" numerator with an "inclusive" denominator. This silently distorted the run rate early in every innings: for ball 2 of an innings (1 run scored off ball 1, 0 balls-before vs. 2 balls-inclusive), the buggy version computed `1 run / 2 balls * 6 = 3.0` instead of the correct `1 run / 1 ball-before * 6 = 6.0` -- roughly half the true rate. Found by `scripts/validate_game_state.py` cross-checking against gagan's own `crr` computed from the *previous* ball's `team_runs`/`team_balls`; now covered by `tests/test_game_state.py::test_run_rate_uses_before_ball_balls_not_including_ball` as a permanent regression test.

## Step 3: runs in the last over, and wickets in the last 5 overs

`runs_last_over` looks at the **previous completed over's** total runs (via a `groupby(...).sum()` per over, then `.shift(1)` to get the *previous* over rather than the current, in-progress one) -- this is a coarser, over-level lookback rather than a strict "before this exact ball" quantity, which is fine since "momentum from the last over" is inherently an over-granularity concept.

`wk_last_5_overs` uses a rolling 30-ball window (5 overs x 6 balls) over the wicket indicator, then subtracts the current ball's own contribution -- same "before this ball" pattern as everything else.

## Step 4: partnership tracking

A "partnership" is the stretch of an innings between two wickets (or from the start of the innings to the first wicket). `wicket_group_id` assigns every ball a group number that increments each time a wicket falls (via cumsum-then-shift, so the ball on which a wicket falls still belongs to the *old* partnership, not the new one that starts on the next ball) -- then `partnership_runs`/`partnership_balls` are just the "before this ball" cumulative runs/balls *within that group*.

## Step 5: in-innings batter/bowler form

How is the specific batter on strike doing *in this innings so far* (not their career average -- their form *today*)? Same "before this ball" cumulative pattern, grouped by `(match_id, innings, batter)` this time instead of just `(match_id, innings)`. Mirrored identically for the bowler (grouped by `(match_id, innings, bowler)`), computing economy rate and wickets taken so far in this spell.

## Step 6: boundaries/dots so far, and the 2nd-innings chase state

Same cumulative "before this ball" pattern again for boundary and dot-ball counts.

For the chase-state features (`runs_required`, `balls_remaining`, `required_rr`), these are all zero for innings 1 (there's no target yet) and only populated for innings 2. `balls_remaining` uses the same `legal_balls_before`-based fix described in Step 2, for the same reason: it must pair correctly with `runs_required` (computed from `score_before`).

## Step 7: toss flag, and assembling the final 24-column matrix

`toss_won_bat` is a simple match-level flag (did the toss winner choose to bat?) broadcast to every ball in that match.

The final `np.stack(...)` assembles all 24 columns in the exact order documented at the top of this module, applying the appropriate normalisation/clipping to each. The `assert` at the end is a cheap sanity check that nothing was accidentally added or dropped from the stack.

In [2]:
def build_game_state_matrix(df: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Compute the (N_balls, 24) game-state matrix for every ball in df
    (project_gagan's cleaned Ball-by-Ball dataframe, one row per delivery).

    Returns the feature matrix plus the input df with the intermediate
    per-ball columns attached (useful for building the win/next-ball/score
    labels alongside the features).
    """
    d = df.sort_values(["match_id", "innings", "over", "ball"]).reset_index(drop=True).copy()

    # NOTE: hrishav's original convention (Cricsheet data) treats no-balls as
    # legal ("over still advances") and only excludes wides. gagan's own
    # ipl_data.xlsx uses a DIFFERENT convention: team_balls excludes both
    # wides AND no-balls (verified empirically: team_balls increments iff
    # extras_wides==0 AND extras_noballs==0, with zero exceptions across all
    # 273,503 deliveries). Matching gagan's convention here -- not hrishav's
    # -- keeps legal_balls_total consistent with team_balls/team_wicket/
    # crr/rrr as used elsewhere in this merged pipeline (src/pipeline.py).
    d["is_legal"] = (d["extras_wides"] == 0) & (d["extras_noballs"] == 0)
    d["phase"] = (d["over"] > 6).astype(int) + (d["over"] > 15).astype(int)

    grp_inn = d.groupby(["match_id", "innings"])

    # legal_balls_total is a POSITION counter (balls bowled so far, including
    # this one) -- deliberately matching gagan's own team_balls semantics
    # (see the is_legal comment above), so it can be compared 1:1 against it.
    # legal_balls_before is the "before this ball" count, used everywhere a
    # ratio needs to pair with score_before/wickets_before (both "before this
    # ball") without mixing a pre-ball numerator against a ball-inclusive
    # denominator -- that mismatch was caught by scripts/validate_game_state.py
    # (run_rate for ball 2 of an innings coming out as score_before/2*6
    # instead of score_before/1*6, i.e. silently halved early in an innings).
    d["legal_balls_total"] = grp_inn["is_legal"].cumsum()
    d["legal_balls_before"] = d["legal_balls_total"] - d["is_legal"].astype(int)
    d["score_before"] = grp_inn["runs_total"].cumsum() - d["runs_total"]
    d["wickets_before"] = grp_inn["is_wicket"].cumsum() - d["is_wicket"]
    d["run_rate"] = d["score_before"] / d["legal_balls_before"].clip(lower=1) * 6

    # Runs in the last completed over: sum of runs_total in the previous over.
    over_runs = d.groupby(["match_id", "innings", "over"])["runs_total"].transform("sum")
    prev_over_map = d.groupby(["match_id", "innings", "over"])["runs_total"].sum().groupby(
        level=[0, 1]).shift(1).fillna(0.0)
    d = d.merge(
        prev_over_map.rename("runs_last_over").reset_index(),
        on=["match_id", "innings", "over"], how="left",
    )
    d["runs_last_over"] = d["runs_last_over"].fillna(0.0)

    # Wickets in the last 5 overs (rolling count over the last 30 legal balls).
    d["wk_last_5_overs"] = (
        grp_inn["is_wicket"]
        .rolling(window=30, min_periods=1)
        .sum()
        .reset_index(level=[0, 1], drop=True)
    ) - d["is_wicket"]

    # Partnership: cumulative runs/balls since the last wicket in this innings.
    wicket_group_id = grp_inn["is_wicket"].cumsum().shift(1).fillna(0)
    d["_partnership_grp"] = wicket_group_id
    part_grp = d.groupby(["match_id", "innings", "_partnership_grp"])
    d["partnership_runs"] = part_grp["runs_total"].cumsum() - d["runs_total"]
    d["partnership_balls"] = part_grp["is_legal"].cumsum() - d["is_legal"].astype(int)

    # In-innings batter strike rate / balls faced so far (before this ball).
    bat_grp = d.groupby(["match_id", "innings", "batter"])
    bat_runs_before = bat_grp["runs_batter"].cumsum() - d["runs_batter"]
    bat_legal_before = bat_grp["is_legal"].cumsum() - d["is_legal"].astype(int)
    d["batter_balls_innings"] = bat_legal_before
    d["batter_sr_innings"] = (bat_runs_before / bat_legal_before.clip(lower=1) * 100).where(
        bat_legal_before > 0, 0.0)

    # In-innings bowler economy / wickets so far (before this ball).
    bowl_grp = d.groupby(["match_id", "innings", "bowler"])
    bowl_runs_before = bowl_grp["runs_total"].cumsum() - d["runs_total"]
    bowl_legal_before = bowl_grp["is_legal"].cumsum() - d["is_legal"].astype(int)
    bowl_wkts_before = bowl_grp["is_wicket"].cumsum() - d["is_wicket"]
    d["bowler_wkts_innings"] = bowl_wkts_before
    d["bowler_econ_innings"] = (bowl_runs_before / bowl_legal_before.clip(lower=1) * 6).where(
        bowl_legal_before > 0, 0.0)

    # Boundaries/dots so far this innings (before this ball).
    d["_is_boundary"] = d["runs_batter"].isin([4, 6]).astype(int)
    d["_is_dot"] = ((d["runs_batter"] == 0) & (d["runs_extras"] == 0)).astype(int)
    grp_inn2 = d.groupby(["match_id", "innings"])
    d["boundaries_before"] = grp_inn2["_is_boundary"].cumsum() - d["_is_boundary"]
    d["dots_before"] = grp_inn2["_is_dot"].cumsum() - d["_is_dot"]

    # 2nd-innings chase state (0 for innings 1). balls_remaining uses
    # legal_balls_before for the same reason run_rate does above: it must
    # pair with runs_required, which is computed from score_before.
    inn2 = d["innings"] == 2
    d["runs_required"] = np.where(inn2, (d["runs_target"] - d["score_before"]).clip(lower=0), 0.0)
    d["balls_remaining"] = np.where(inn2, (120 - d["legal_balls_before"]).clip(lower=1), 0.0)
    d["required_rr"] = np.where(inn2, d["runs_required"] / np.clip(d["balls_remaining"], 1, None) * 6, 0.0)

    # Toss: did the toss winner choose to bat (match-level, broadcast per ball).
    d["toss_won_bat"] = (d["toss_decision"] == "bat").astype(float)

    lb = d["legal_balls_before"].clip(lower=1)
    inn2_f = inn2.astype(float)
    phase = d["phase"]

    X = np.stack([
        d["over"] / 20.0,
        d["legal_balls_total"] / 120.0,
        inn2_f,
        (phase == 0).astype(float),
        (phase == 1).astype(float),
        (phase == 2).astype(float),
        d["score_before"] / 250.0,
        d["wickets_before"] / 10.0,
        (d["run_rate"] / 15.0).clip(upper=1.0),
        d["runs_last_over"] / 30.0,
        d["wk_last_5_overs"] / 5.0,
        d["partnership_runs"] / 150.0,
        d["partnership_balls"] / 60.0,
        d["batter_sr_innings"] / 200.0,
        d["batter_balls_innings"] / 120.0,
        (d["bowler_econ_innings"] / 20.0).clip(upper=1.0),
        d["bowler_wkts_innings"] / 10.0,
        d["boundaries_before"] / lb,
        d["dots_before"] / lb,
        d["runs_required"] / 200.0,
        d["balls_remaining"] / 120.0,
        (d["required_rr"] / 36.0).clip(upper=1.0),
        d["toss_won_bat"],
        inn2_f,
    ], axis=1).astype(np.float32)

    assert X.shape[1] == GAME_STATE_DIM
    return X, d